In [7]:
!pip install --upgrade "plotly>=6.1.1" kaleido pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 66.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 80.8 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: plotly
    Found existing installation: plotly 5.24.1
    Uninstalling plotly-5.24.1:
      Successfully uninstalled plotly-5.24.1
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully uninstalled pandas-2.2.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
dask-cudf-cu12 25.2.2 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.2 which is incompatible.
cudf-cu12 25.2.2 requires pandas<2.2.4dev0,>=2.0, but you have

In [19]:
import os
import glob
import pandas as pd
import plotly.graph_objects as go


In [20]:
c = ["/kaggle/input/nscnjssnc/EEG and ECG data_02_raw.csv"]
p = [x for x in c if os.path.exists(x)] or glob.glob("**/EEG*ECG*raw*.csv", recursive=True)
if not p:
    raise FileNotFoundError("CSV missing")

f = pd.read_csv(p[0], comment="#")
tcol = next((x for x in f.columns if x.strip().lower() == "time"), None)
if not tcol:
    raise ValueError("Missing Time")


In [21]:
E = {"Fz","Cz","P3","C3","F3","F4","C4","P4","Fp1","Fp2","T3","T4","T5","T6","O1","O2","F7","F8","A1","A2","Pz"}

eeg = [x for x in f.columns if x.strip() in E]
ecg = list({"X1:LEOG","X2:REOG"} & set(f.columns))
cm  = ["CM"] if "CM" in f.columns else []

cols = [tcol] + eeg + ecg + cm
d = f[cols].copy()

for x in d.columns:
    if x != tcol:
        d[x] = pd.to_numeric(d[x], errors="coerce").astype("float32")


In [22]:
t = d[tcol]
eegp = [x for x in d.columns if x not in (tcol, "CM") and not x.startswith("X")]
ecgp = [x for x in d.columns if x.startswith("X")]
has_cm = "CM" in d.columns

fig = go.Figure()
for x in eegp:
    fig.add_trace(go.Scatter(x=t, y=d[x], mode="lines", name=x, yaxis="y1"))
for x in ecgp:
    fig.add_trace(go.Scatter(x=t, y=d[x], mode="lines", name=f"{x}(mV)", yaxis="y2"))
if has_cm:
    fig.add_trace(go.Scatter(x=t, y=d["CM"], mode="lines", name="CM(ref)", yaxis="y2"))

fig.update_layout(
    title="EEG µV + ECG mV",
    xaxis=dict(title="Time(s)", rangeslider=dict(visible=True), type="linear"),
    yaxis=dict(title="EEG µV"),
    yaxis2=dict(title="ECG / CM mV", overlaying="y", side="right"),
    hovermode="x unified",
    legend=dict(orientation="h", y=1.08)
)


In [23]:
n1 = len(eegp)
n2 = len(fig.data) - n1

allv  = [True]*(n1+n2)
eonly = [True]*n1 + [False]*n2
ronly = [False]*n1 + [True]*n2

fig.update_layout(
    updatemenus=[dict(
        type="dropdown", x=0.01, y=1.2, showactive=True,
        buttons=[
            dict(label="All", method="update", args=[{"visible": allv}]),
            dict(label="EEG", method="update", args=[{"visible": eonly}]),
            dict(label="ECG+CM", method="update", args=[{"visible": ronly}])
        ]
    )]
)
fig.show()
